In [ ]:
!pip install kagglehub transformers torchvision scikit-learn grad-cam -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 75.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ryandpark/fruit-quality-classification")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'fruit-quality-classification' dataset.
Path to dataset files: /kaggle/input/fruit-quality-classification


In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    print("Folders:", dirs[:10])
    print("Files:", files[:3])
    print("-" * 80)

/kaggle/input/fruit-quality-classification
Folders: ['Good Quality_Fruits', 'Mixed Qualit_Fruits', 'Bad Quality_Fruits']
Files: []
--------------------------------------------------------------------------------
/kaggle/input/fruit-quality-classification/Good Quality_Fruits
Folders: ['Apple_Good', 'Pomegranate_Good', 'Guava_Good', 'Banana_Good', 'Lime_Good', 'Orange_Good']
Files: []
--------------------------------------------------------------------------------
/kaggle/input/fruit-quality-classification/Good Quality_Fruits/Apple_Good
Folders: []
Files: ['20190809_122415.jpg', 'IMG_9450.JPG', '20190809_161713.jpg']
--------------------------------------------------------------------------------
/kaggle/input/fruit-quality-classification/Good Quality_Fruits/Pomegranate_Good
Folders: []
Files: ['20190820_144207_23160.jpg', '20190820_144620.jpg', '20190820_150756_24344.jpg']
--------------------------------------------------------------------------------
/kaggle/input/fruit-quality-classi

In [ ]:
import os
import shutil
import random
from pathlib import Path

source_root = path   # kagglehub path

binary_root = "/content/binary_dataset"

# Create folders
for split in ["train", "test"]:
    os.makedirs(f"{binary_root}/{split}/healthy", exist_ok=True)
    os.makedirs(f"{binary_root}/{split}/defective", exist_ok=True)

healthy_files = []
defective_files = []

# Good -> Healthy
good_root = os.path.join(source_root, "Good Quality_Fruits")

for fruit_folder in os.listdir(good_root):
    fruit_path = os.path.join(good_root, fruit_folder)

    for img in os.listdir(fruit_path):
        healthy_files.append(os.path.join(fruit_path, img))

# Bad -> Defective
bad_root = os.path.join(source_root, "Bad Quality_Fruits")

for fruit_folder in os.listdir(bad_root):
    fruit_path = os.path.join(bad_root, fruit_folder)

    for img in os.listdir(fruit_path):
        defective_files.append(os.path.join(fruit_path, img))

# Mixed -> Defective
mixed_root = os.path.join(source_root, "Mixed Qualit_Fruits")

for fruit_folder in os.listdir(mixed_root):
    fruit_path = os.path.join(mixed_root, fruit_folder)

    for img in os.listdir(fruit_path):
        defective_files.append(os.path.join(fruit_path, img))

print("Healthy:", len(healthy_files))
print("Defective:", len(defective_files))

Healthy: 11664
Defective: 7862


In [ ]:
random.seed(42)

random.shuffle(healthy_files)
random.shuffle(defective_files)

healthy_split = int(0.8 * len(healthy_files))
defective_split = int(0.8 * len(defective_files))

healthy_train = healthy_files[:healthy_split]
healthy_test = healthy_files[healthy_split:]

defective_train = defective_files[:defective_split]
defective_test = defective_files[defective_split:]

print(len(healthy_train), len(healthy_test))
print(len(defective_train), len(defective_test))

9331 2333
6289 1573


In [ ]:
def copy_files(file_list, destination):
    for file_path in file_list:
        filename = os.path.basename(file_path)
        shutil.copy(file_path,
                    os.path.join(destination, filename))

copy_files(
    healthy_train,
    f"{binary_root}/train/healthy"
)

copy_files(
    healthy_test,
    f"{binary_root}/test/healthy"
)

copy_files(
    defective_train,
    f"{binary_root}/train/defective"
)

copy_files(
    defective_test,
    f"{binary_root}/test/defective"
)

print("Dataset created successfully.")

Dataset created successfully.


In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder("/content/binary_dataset/train")

print(dataset.classes)
print("Number of training images:", len(dataset))

['defective', 'healthy']
Number of training images: 15262


In [ ]:
import os

train_healthy = len(os.listdir("/content/binary_dataset/train/healthy"))
train_defective = len(os.listdir("/content/binary_dataset/train/defective"))

test_healthy = len(os.listdir("/content/binary_dataset/test/healthy"))
test_defective = len(os.listdir("/content/binary_dataset/test/defective"))

print("TRAIN")
print("Healthy:", train_healthy)
print("Defective:", train_defective)

print("\nTEST")
print("Healthy:", test_healthy)
print("Defective:", test_defective)

TRAIN
Healthy: 8973
Defective: 6289

TEST
Healthy: 2305
Defective: 1573


In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
test_transforms = transforms.Compose([
    transforms.Resize(256),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder(
    "/content/binary_dataset/train",
    transform=train_transforms
)

test_dataset = ImageFolder(
    "/content/binary_dataset/test",
    transform=test_transforms
)

print(train_dataset.classes)

['defective', 'healthy']


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
!pip install transformers -q

In [ ]:
from transformers import SwinForImageClassification

model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224",
    num_labels=2,
    id2label={
        0:"defective",
        1:"healthy"
    },
    label2id={
        "defective":0,
        "healthy":1
    },
    ignore_mismatched_sizes=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
for param in model.swin.parameters():
    param.requires_grad = False

In [ ]:
model.to(device)

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (attention): SwinAttention(
                (q_proj): Linear(in_features=96, out_features=96, bias=True)
                (k_proj): Linear(in_features=96, out_features=96, bias=True)
                (v_proj): Linear(in_features=96, out_features=96, bias=True)
                (o_proj): Linear(in_features=96, out_features=96, bias=True)
                (relative_position_bias): SwinRelativePositionBias()
              )
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
         

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

epochs = 8

scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
from tqdm.auto import tqdm
import copy
import torch.nn.functional as F

In [ ]:
best_accuracy = 0.0
best_model_weights = None

In [ ]:
for epoch in range(epochs):

    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"{'='*50}")

    # -------------------
    # TRAINING
    # -------------------
    model.train()

    running_loss = 0.0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(
                images,
                labels=labels
            )

            loss = outputs.loss

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    scheduler.step()

    # -------------------
    # TESTING
    # -------------------
    model.eval()

    correct = 0
    total = 0

    test_loss = 0.0

    with torch.no_grad():

        for images, labels in tqdm(test_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(
                images,
                labels=labels
            )

            loss = outputs.loss

            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)

    accuracy = correct / total

    print(
        f"Train Loss: {avg_train_loss:.4f}"
    )

    print(
        f"Test Loss: {avg_test_loss:.4f}"
    )

    print(
        f"Accuracy: {accuracy*100:.2f}%"
    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model_weights = copy.deepcopy(
            model.state_dict()
        )

        print(
            "New Best Model Saved"
        )


Epoch 1/8


  0%|          | 0/477 [00:00<?, ?it/s]

  0%|          | 0/122 [00:00<?, ?it/s]

Train Loss: 0.1444
Test Loss: 0.0976
Accuracy: 96.67%
New Best Model Saved

Epoch 2/8


  0%|          | 0/477 [00:00<?, ?it/s]

...

  0%|          | 0/122 [00:00<?, ?it/s]

Train Loss: 0.0507
Test Loss: 0.0620
Accuracy: 97.83%

Epoch 8/8


  0%|          | 0/477 [00:00<?, ?it/s]

  0%|          | 0/122 [00:00<?, ?it/s]

Train Loss: 0.0511
Test Loss: 0.0613
Accuracy: 97.99%
New Best Model Saved


In [ ]:
model.load_state_dict(best_model_weights)

torch.save(
    model.state_dict(),
    "/content/swin_fruit_binary.pth"
)

print("Best model saved.")

Best model saved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/swin_fruit_binary.pth"
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

y_true = []
y_pred = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        preds = outputs.logits.argmax(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "defective",
            "healthy"
        ]
    )
)

print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

   defective       0.96      0.99      0.98      1573
     healthy       0.99      0.97      0.98      2305

    accuracy                           0.98      3878
   macro avg       0.98      0.98      0.98      3878
weighted avg       0.98      0.98      0.98      3878

[[1553   20]
 [  58 2247]]


In [ ]:
from google.colab import files

# Download the saved model to your local machine
files.download("/content/swin_fruit_binary.pth")